In [66]:
import pandas as pd
import numpy as np
import pickle
from lightgbm import LGBMRegressor
MODELS_CACHE = {}

In [67]:
df = pd.read_csv("full_data.csv")

In [68]:
pitchers = list(df["pitcherId"].unique())
pitches = ("FA", "SL", "CH", "CU", "FC", "SI", "FS")
lr = ["L", "R"]

features = ['inducedVertBreak', 'horzBreak', 'extension', 'relX', 'relZ', 'releaseVelocity', 'spinRate']
target = ["rv"]

test = df[df["year"] == 2026]

In [69]:
test = test[test["pitchType"].isin(pitches)]

In [70]:
def pred_rv(test, features, target, pitch, side):
    if (pitch, side) not in MODELS_CACHE:
        with open(f'models/26_{pitch}_{side}.pkl', 'rb') as f:
            MODELS_CACHE[(pitch, side)] = pickle.load(f)

    model = MODELS_CACHE[(pitch, side)]
    X_test = test[features]
    predictions = model.predict(X_test) 
    
    return pd.Series(predictions, index=test.index)

In [71]:
rows = []
i=1

for index, row in test.iterrows():
    pitch = row["pitchType"]
    side = row["pitcherHand"]

    test = pd.DataFrame([row])
    
    test["xrv"] = float(pred_rv(test, features, target, pitch, side).iloc[0]) 
    rows.append(test)

    if i % 100000 == 0:
        print(i)

    i = i+1

100000
200000
300000
400000
500000
600000
700000
800000
900000
1000000
1100000
1200000
1300000
1400000
1500000
1600000
1700000


In [72]:
new = pd.concat(rows)

In [98]:
stuff = new.groupby(["pitcherId", "pitchType"]).agg(
    Pitcher = ("Pitcher", "first"),
    year = ("year", "mean"),
    team = ("pitchingTeam", "first"),
    Hand = ("pitcherHand", "first"),
    Count = ("pitchType", "count"),
    RelSpeed = ("releaseVelocity", "mean"),
    IVB = ("inducedVertBreak", "mean"),
    HB = ("horzBreak", "mean"),
    SpinRate = ("spinRate", "mean"),
    RelSide = ("relX", "mean"),
    RelHeight = ("relZ", "mean"),
    Extension = ("extension", "mean"),
    xrv = ("xrv", "mean")
).reset_index()

In [99]:
stuff = stuff[["Pitcher", "pitcherId", "team", "pitchType", "Hand", "Count", "RelSpeed", "IVB", "HB", "SpinRate", "RelSide", "RelHeight", "Extension", "xrv"]]
stuff = stuff[stuff["Count"] >= 25]
#stuff["RelSide"] = -stuff["RelSide"]

In [102]:
stuff = stuff.rename(columns={'pitchType': 'PitchType'})

In [103]:
stuff[stuff["team"] == "URI"].sort_values("xrv")

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv
12264,P. Aikens,1299756340,URI,SL,L,183,72.772005,5.277696,16.742062,2663.800719,-2.821260,4.682106,4.208979,0.055702
10660,J. Kopetski,1269533184,URI,FC,L,66,76.672211,7.918584,3.572704,1709.591642,-2.054742,5.623629,6.145470,0.055931
17981,M. Santos,1487988943,URI,SL,R,198,81.850110,-0.354423,-16.391414,2649.944813,2.655238,5.025247,5.261400,0.062804
3225,J. Sabbath,1110853376,URI,SL,R,253,78.684421,4.809984,-11.676591,2367.322806,1.795760,5.682230,5.824940,0.064286
2867,A. Jones,1101042688,URI,CU,R,40,76.407526,-17.789101,-10.938031,2581.275308,1.020377,5.843453,5.316856,0.065423
15995,J. Cullen,1358089441,URI,SL,R,102,83.205071,2.313787,-5.057162,2614.971084,1.496568,5.838506,5.521894,0.066373
1836,E. Maloney,1085415168,URI,CH,R,105,75.723958,9.951533,9.388448,1476.909301,0.044177,6.945864,4.893052,0.066477
15991,J. Cullen,1358089441,URI,CU,R,63,76.407241,-13.357738,-14.064611,2764.255015,1.217194,6.194429,5.257873,0.069451
4403,L. Lavigueur,1143021056,URI,SL,R,130,78.692538,-5.031048,-19.986650,2549.220573,1.497158,5.895682,5.287887,0.069590
14320,C. Maloney,1336247552,URI,SL,R,65,82.426083,-3.094618,-6.825434,2565.914300,2.303367,5.767832,5.328540,0.070053


In [105]:
from scipy import stats

#df['zscore'] = stats.zscore(df['column'])
stuff["Stuff+"] = 100 + (-10 * stuff.groupby("PitchType")["xrv"].transform(stats.zscore))
stuff["Stuff+"] = stuff["Stuff+"].round().astype(int)

In [106]:
stuff.sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26463,D. Volantis,1979617275,TEX,CU,L,376,81.379708,-12.068357,3.341951,2586.700135,-2.084254,6.801370,5.220089,-0.005837,162
12323,H. Dietz,1301090561,ARK,CU,L,135,80.181215,-14.551537,7.793933,2741.135135,-2.183739,6.723800,5.275731,0.004563,154
14294,B. Bryans,1335999920,JVST,SL,L,136,87.635811,2.384733,9.565049,2289.941065,-4.606932,4.447350,5.851318,0.018837,147
3591,A. Buczkowski,1123936875,CIN,SL,R,332,81.022915,10.764309,-12.433597,2579.659444,5.365934,3.633378,5.298328,0.023982,143
5085,G. Helsel,1162012672,XAV,FC,L,160,78.239304,7.752828,7.783435,2179.948371,-2.486020,6.621737,5.370563,0.009470,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13760,C. Tryba,1324981760,UCSB,CH,L,88,81.528135,1.568656,-21.423276,2005.116134,-1.627252,4.252699,5.580087,0.163091,47
5628,J. Rosenblitt,1164786998,OAK,CU,R,41,68.313938,5.793701,-7.111767,1836.721304,0.554607,5.836005,5.587606,0.152272,41
7897,G. Alveti,1207758080,PEAY,CH,R,33,78.261204,-7.603355,8.859135,1850.506912,4.243020,2.876326,6.152433,0.172049,40
2778,T. Beard,1100603904,FSU,SL,L,149,84.998237,2.992909,-2.781686,1402.020891,-0.510950,6.688386,5.312259,0.150676,39


In [107]:
stuff.sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26463,D. Volantis,1979617275,TEX,CU,L,376,81.379708,-12.068357,3.341951,2586.700135,-2.084254,6.801370,5.220089,-0.005837,162
12323,H. Dietz,1301090561,ARK,CU,L,135,80.181215,-14.551537,7.793933,2741.135135,-2.183739,6.723800,5.275731,0.004563,154
14294,B. Bryans,1335999920,JVST,SL,L,136,87.635811,2.384733,9.565049,2289.941065,-4.606932,4.447350,5.851318,0.018837,147
3591,A. Buczkowski,1123936875,CIN,SL,R,332,81.022915,10.764309,-12.433597,2579.659444,5.365934,3.633378,5.298328,0.023982,143
5085,G. Helsel,1162012672,XAV,FC,L,160,78.239304,7.752828,7.783435,2179.948371,-2.486020,6.621737,5.370563,0.009470,141
167,D. Hamilton,1008376863,CARK,SI,R,62,90.468106,17.620931,17.544432,2455.711160,2.492114,5.122982,5.622249,0.049713,139
12584,C. Alderman,1306683393,MTU,SI,L,25,90.076829,10.768046,-7.019397,2106.451048,-2.171599,6.431547,6.617203,0.050084,139
22354,J. Lucas,1734064400,TROY,FC,L,29,77.319238,5.877769,10.628479,2427.284920,-2.469060,5.269765,5.855865,0.015558,138
21256,B. Trosclair,1674977859,NWST,CU,L,29,78.090700,-3.240524,10.456365,2977.784566,-2.155311,5.557400,6.474932,0.026794,137
6890,J. Mitchell,1194506240,BING,CH,R,126,85.012640,7.947642,14.445318,1370.873945,0.084755,6.661576,5.877077,0.043671,136


In [108]:
stuff[stuff["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
1836,E. Maloney,1085415168,URI,CH,R,105,75.723958,9.951533,9.388448,1476.909301,0.044177,6.945864,4.893052,0.066477,119
15992,J. Cullen,1358089441,URI,FA,R,341,91.595910,21.455902,10.417795,2257.518696,1.108587,6.102822,6.004221,0.073752,119
12264,P. Aikens,1299756340,URI,SL,L,183,72.772005,5.277696,16.742062,2663.800719,-2.821260,4.682106,4.208979,0.055702,117
10660,J. Kopetski,1269533184,URI,FC,L,66,76.672211,7.918584,3.572704,1709.591642,-2.054742,5.623629,6.145470,0.055931,115
7660,C. Grotyohann,1203468032,URI,FA,R,173,90.735434,20.985878,8.971219,2186.473280,1.877272,6.187250,5.882359,0.081169,115
17979,M. Santos,1487988943,URI,FA,R,192,92.588882,16.130212,7.259698,2465.108502,2.400913,5.323777,5.615804,0.084572,113
17981,M. Santos,1487988943,URI,SL,R,198,81.850110,-0.354423,-16.391414,2649.944813,2.655238,5.025247,5.261400,0.062804,111
9514,C. Johnston,1241946624,URI,CH,R,146,80.239595,9.822503,13.873437,1746.363225,0.742984,6.057004,6.446628,0.077630,111
14317,C. Maloney,1336247552,URI,FA,R,110,90.298077,19.079510,10.144955,2198.203184,1.944296,5.966695,5.874133,0.088917,111
2868,A. Jones,1101042688,URI,FA,R,143,89.682956,21.046223,7.763579,2156.832315,1.054245,5.835070,5.834861,0.091581,110


In [109]:
stuff[stuff["PitchType"] == "CH"].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,PitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
6890,J. Mitchell,1194506240,BING,CH,R,126,85.012640,7.947642,14.445318,1370.873945,0.084755,6.661576,5.877077,0.043671,136
12007,G. Naess,1296842752,CP,CH,R,241,75.690942,7.757739,19.224069,2097.406187,1.567333,6.460146,5.758553,0.045363,135
7797,D. Jackson,1205651242,UCSB,CH,L,148,76.599535,3.314885,-20.151771,2558.675981,-1.580773,5.827232,5.795063,0.047916,133
12706,B. Purcell,1307080960,FSU,CH,R,85,85.429949,4.985430,15.758260,1380.141620,0.350854,6.753516,5.108490,0.048031,133
7995,J. DeCaro,1207928320,UNC,CH,R,120,82.469379,8.496792,17.775047,2277.880300,2.513911,6.319347,5.452918,0.053263,129
19664,C. Chatterton,1584179280,AUB,CH,R,114,75.909575,8.770764,14.380997,2329.557059,0.629131,5.905677,5.512704,0.055339,128
7221,R. Marohn,1197348608,NCST,CH,L,154,84.907417,5.094050,-16.728264,1619.743751,-3.761042,5.360392,6.267285,0.054969,128
25065,T. Martin,1893153714,ME,CH,R,34,75.830455,9.776340,16.335410,2038.150677,0.551503,6.292393,5.808309,0.055073,128
12517,S. Hall,1305069568,TXST,CH,R,52,84.145973,5.787494,13.191422,1032.159339,0.142647,6.533018,5.371311,0.055715,127
20485,J. Barkoff,1626135626,ORE,CH,L,32,79.942131,4.032901,-19.086768,2233.619709,-1.191728,5.806893,5.882110,0.055818,127


In [110]:
stuff.to_csv("stuff.csv")

In [84]:
stuff[stuff["Count"] > 200].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,pitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
26463,D. Volantis,1979617275,TEX,CU,L,376,81.379708,-12.068357,3.341951,2586.700135,-2.084254,6.801370,5.220089,-0.005837,162
3591,A. Buczkowski,1123936875,CIN,SL,R,332,81.022915,10.764309,-12.433597,2579.659444,5.365934,3.633378,5.298328,0.023982,143
12007,G. Naess,1296842752,CP,CH,R,241,75.690942,7.757739,19.224069,2097.406187,1.567333,6.460146,5.758553,0.045363,135
12325,H. Dietz,1301090561,ARK,FC,L,364,85.185284,1.097770,1.905836,2472.532920,-2.403032,6.522148,5.570758,0.026204,132
17621,S. Sandford,1464054215,FLA,FA,R,222,95.182954,17.826385,10.717254,2306.471496,3.329584,5.901099,6.311194,0.048326,132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14234,J. Lessman,1334299648,UIC,CU,L,202,75.522496,-10.949484,9.288853,2166.790101,-1.950915,5.051417,5.679374,0.111960,72
22833,L. Doll,1760080732,MRSH,CU,L,225,72.555626,-7.758700,11.696452,2126.345612,-2.869837,5.101466,6.430146,0.112792,71
10059,N. Chapman,1257329152,UNCG,CH,L,213,81.580937,5.450538,-10.668859,1577.084685,-2.760667,5.440038,5.159483,0.131984,70
2936,L. Nichols,1104573440,LT,SI,L,293,88.291664,5.713710,-13.349514,1904.393867,-1.372755,5.224689,6.524327,0.160617,69


In [86]:
stuff[(stuff["Count"] > 50) & (stuff["pitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,pitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
5061,J. Music,1161348114,CAMP,FA,R,194,92.104651,21.624209,13.339871,2209.389954,3.530360,5.966601,5.683935,0.045460,134
20095,S. Garcia,1602557566,LSU,FA,L,157,92.719318,21.283184,-10.994845,2441.646707,-1.623876,6.250958,6.137295,0.046986,133
24618,J. Barberi,1862463014,FLA,FA,R,233,97.544816,20.973145,11.005159,2528.092311,1.833742,6.330433,5.433847,0.049301,132
17621,S. Sandford,1464054215,FLA,FA,R,222,95.182954,17.826385,10.717254,2306.471496,3.329584,5.901099,6.311194,0.048326,132
7270,S. Matson,1198395392,USC,FA,L,383,90.299701,16.577497,-15.990251,2311.717114,-1.662029,4.801586,6.152693,0.049740,131
7223,R. Marohn,1197348608,NCST,FA,L,381,91.792761,17.194621,-13.966061,2262.851961,-3.642764,5.568628,6.151826,0.049767,131
19186,J. Nottingham,1557710177,UGA,FA,R,156,95.981355,21.045768,12.185054,2556.920229,1.535647,6.177304,5.989528,0.051245,131
5312,W. Whelan,1162337024,MINN,FA,L,136,92.177543,15.083399,-16.511609,2257.656539,-3.674303,5.279304,5.708430,0.049558,131
14918,P. Manca,1342167297,FSU,FA,L,84,90.799642,22.895341,-9.065927,2388.202370,-1.720101,6.708724,5.786204,0.052081,130
4221,J. Raab,1139725824,GTWN,FA,R,252,92.604546,22.756290,11.202625,2514.467266,2.102659,6.136396,6.112460,0.053197,130


In [20]:
stuff[(stuff["PitcherThrows"] == "R") & (stuff["PitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
490,J. Music,1161348114,CAMP,FA,R,173,21.628388,13.648429,5.659460,-3.559814,5.955501,91.997280,2213.872287,0.034663,135
11971,J. Nottingham,1557710177,UGA,FA,R,145,21.125900,12.154075,6.015362,-1.536138,6.166429,95.914997,2560.459596,0.038500,133
4823,B. Frederick,1158279680,TENN,FA,R,297,1.572949,7.436813,5.701303,-2.901594,1.640074,82.359681,2168.243243,0.038908,133
2388,J. Raab,1139725824,GTWN,FA,R,207,23.336993,11.135619,6.086409,-2.208609,6.104525,92.491013,2511.967254,0.041329,132
4345,S. Sandford,1464054215,FLA,FA,R,192,17.901868,10.574753,6.306502,-3.370593,5.863227,95.055346,2300.540448,0.046436,130
10072,C. Clark,1956935142,USM,FA,R,331,21.549512,9.081232,6.479228,-1.775685,5.490115,93.422269,2266.997569,0.048969,129
4355,J. Barberi,1862463014,FLA,FA,R,212,20.894572,10.942414,5.447520,-1.818514,6.297267,97.477952,2525.121508,0.047150,129
6375,A. Roblez,1180787200,ORST,FA,R,118,24.373893,2.416431,6.217429,-0.757470,6.694467,91.422169,2342.904456,0.050728,128
1894,D. Sheerin,1213263616,LSU,FA,R,309,14.685014,16.386464,6.015708,-3.526091,5.499113,96.379109,2446.678971,0.053563,127
10565,L. ODonnell,1691990270,USF,FA,R,97,21.633855,9.905414,6.210631,-1.165482,6.300660,95.154776,2477.496614,0.052742,127


In [21]:
import pandas as pd
import numpy as np
stuff = pd.read_csv("stuff.csv")
pitches = list(stuff["PitchType"].unique())

In [87]:
rows = []
ids = list(stuff["pitcherId"].unique())

for i in ids:
    x = stuff[stuff["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["team"] = x["team"].iloc[0]
    dx["Hand"] = x["Hand"].iloc[0]
    for pt in list(stuff["pitchType"].unique()):
        if pt in list(x["pitchType"].unique()):
            dx["Stf+ " + pt] = x[x["pitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["pitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)

In [88]:
table = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table.columns if col.startswith("Stf+")]
table[stf_cols] = table[stf_cols].astype("Int64")

table["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table.columns and ct_col in table.columns:
        mask = table[col].notna()
        table.loc[mask, "Stuff+"] += (
            table.loc[mask, col] * (table.loc[mask, ct_col] / table.loc[mask, "Total"])
        )

table["Stuff+"] = table["Stuff+"].round().astype(int)
table = table[["Pitcher", "pitcherId", "team", "Total", "Hand", "Stf+ FA", "Stf+ SL", "Stf+ CU", "Stf+ FC", "Stf+ SI", "Stf+ CH", "Stf+ FS", "Stuff+"]]

In [89]:
table[table["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
3199,M. Santos,1487988943,URI,390,R,113,111,<NA>,<NA>,<NA>,<NA>,<NA>,112
2856,J. Cullen,1358089441,URI,726,R,119,108,104,104,<NA>,94,<NA>,110
2562,C. Maloney,1336247552,URI,175,R,111,105,<NA>,<NA>,<NA>,<NA>,<NA>,109
1369,C. Grotyohann,1203468032,URI,296,R,115,105,<NA>,96,<NA>,<NA>,<NA>,108
513,A. Jones,1101042688,URI,262,R,110,104,107,103,<NA>,<NA>,<NA>,108
579,J. Sabbath,1110853376,URI,768,R,106,110,99,<NA>,<NA>,94,<NA>,106
2197,W. Creed,1299086745,URI,68,L,106,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,106
2204,P. Aikens,1299756340,URI,355,L,94,117,<NA>,<NA>,<NA>,<NA>,<NA>,106
2731,B. Perry,1342168540,URI,56,R,104,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,104
327,E. Maloney,1085415168,URI,482,R,99,101,94,<NA>,<NA>,119,<NA>,103


In [90]:
table[table["Total"] >= 500].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
4702,D. Volantis,1979617275,TEX,967,L,101,<NA>,162,118,102,93,<NA>,129
1287,R. Marohn,1197348608,NCST,638,L,131,<NA>,113,<NA>,<NA>,128,<NA>,127
2216,H. Dietz,1301090561,ARK,1175,L,112,129,154,132,<NA>,<NA>,<NA>,125
641,A. Buczkowski,1123936875,CIN,595,R,98,143,<NA>,109,<NA>,<NA>,<NA>,124
1296,S. Matson,1198395392,USC,580,L,131,105,<NA>,<NA>,<NA>,<NA>,<NA>,122
3299,W. Sanford,1518897568,ORE,1166,R,127,103,118,<NA>,<NA>,108,<NA>,122
780,L. Peterson,1142994176,FLA,1217,R,124,128,114,109,<NA>,104,<NA>,121
1222,Z. Peters,1193094400,VCU,739,R,125,117,107,<NA>,<NA>,<NA>,<NA>,120
2023,M. Edwards,1279382784,USC,1206,L,123,111,120,<NA>,<NA>,108,<NA>,120
1796,E. Nachtsheim,1256101932,MCNS,1087,R,127,103,93,82,<NA>,105,<NA>,120


In [56]:
table[table["Total"] >= 100].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
4606,C. Van Ameyde,1970680555,ND,165,R,<NA>,<NA>,<NA>,<NA>,138,<NA>,<NA>,138
3357,J. Nottingham,1557710177,UGA,145,R,130,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,130
927,W. Whelan,1162337024,MINN,111,L,129,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,129
884,J. Music,1161348114,CAMP,255,R,134,123,<NA>,<NA>,<NA>,109,<NA>,128
1663,H. Leffew,1239346432,TEX,216,L,130,<NA>,121,<NA>,<NA>,<NA>,<NA>,126
739,J. Raab,1139725824,GTWN,268,R,132,105,<NA>,<NA>,<NA>,<NA>,107,126
4624,D. Volantis,1979617275,TEX,794,L,100,<NA>,155,116,105,95,<NA>,125
2243,B. Purcell,1307080960,FSU,393,R,127,115,<NA>,<NA>,<NA>,135,<NA>,125
4299,J. Barberi,1862463014,FLA,413,R,131,118,<NA>,<NA>,<NA>,<NA>,<NA>,125
3089,S. Sandford,1464054215,FLA,268,R,131,108,<NA>,<NA>,<NA>,<NA>,<NA>,124


In [61]:
table[table["Total"] >= 100].sort_values("Stf+ FS", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
763,J. Noot,1142711296,LSU,117,R,94,<NA>,<NA>,<NA>,107,<NA>,122,108
879,L. Earnhardt,1160764160,WIN,705,L,102,98,106,<NA>,<NA>,117,119,105
1248,C. Estridge,1196802304,WVU,372,R,104,99,<NA>,<NA>,100,<NA>,118,102
2509,J. Giordano,1334554354,RICH,819,L,107,102,104,<NA>,<NA>,104,117,106
2973,P. Green,1424021277,MIZZ,171,R,111,112,<NA>,<NA>,<NA>,<NA>,116,112
1365,O. Marsh,1205184939,BEL,227,R,94,101,<NA>,<NA>,<NA>,<NA>,115,100
3359,K. Sweum,1558099259,GONZ,918,L,112,126,<NA>,100,<NA>,93,113,113
177,D. Brousseau,1055997641,BYU,318,R,96,<NA>,<NA>,<NA>,104,111,113,102
613,L. Kenny,1120804043,UOWG,597,R,100,105,<NA>,103,<NA>,112,113,105
20,K. Althoff,1006521813,CHSO,167,R,87,<NA>,<NA>,116,<NA>,<NA>,111,98


In [91]:
table[table["team"] == "MRMK"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
2477,Z. Broderick,1327271424,MRMK,128,L,105,<NA>,<NA>,<NA>,<NA>,97,<NA>,103
4096,S. Franco,1769668118,MRMK,75,L,100,<NA>,94,<NA>,<NA>,<NA>,<NA>,98
4494,J. Nichols,1907374041,MRMK,153,R,92,101,<NA>,100,<NA>,<NA>,<NA>,96
2245,C. Barker,1304701440,MRMK,190,R,96,93,<NA>,96,<NA>,<NA>,<NA>,95
2697,N. Hunkele,1342167952,MRMK,227,R,95,77,<NA>,<NA>,<NA>,108,<NA>,94
2979,C. Kiesiner,1402009816,MRMK,119,R,93,93,<NA>,<NA>,<NA>,<NA>,<NA>,93
2279,B. Loper,1307046400,MRMK,90,R,88,90,<NA>,<NA>,<NA>,100,<NA>,92
2929,O. Barkal,1384801355,MRMK,183,R,85,94,<NA>,<NA>,<NA>,<NA>,<NA>,92
1116,C. Cassidy,1183063552,MRMK,38,L,91,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,91
1484,J. Borsari,1213486080,MRMK,70,R,89,92,<NA>,<NA>,<NA>,<NA>,<NA>,91


In [60]:
stuff[stuff["pitcherId"] == 1304701440]

,Pitcher,pitcherId,team,pitchType,Hand,Count,RelSpeed,IVB,HB,SpinRate,RelSide,RelHeight,Extension,xrv,Stuff+
12208,C. Barker,1304701440,MRMK,FA,R,87,88.475359,17.802189,11.652672,2141.249840,1.426901,6.342836,6.193289,0.117388,96
12209,C. Barker,1304701440,MRMK,FC,R,32,82.083254,8.375591,0.372366,1992.432315,1.556288,6.185961,6.197989,0.093367,94
12211,C. Barker,1304701440,MRMK,SL,R,70,77.022350,-0.921824,-5.590443,2132.531281,1.555277,6.218094,5.940046,0.086247,92


In [92]:
table.to_csv("stuff_table.csv")

In [63]:
table[table["team"] == "ECU"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
2840,C. Weber,1369039521,ECU,215,R,118,122,<NA>,<NA>,<NA>,<NA>,<NA>,119
436,G. Van Kempen,1099204864,ECU,428,R,114,124,<NA>,113,<NA>,124,<NA>,117
388,L. Payne,1093095680,ECU,305,L,117,<NA>,<NA>,<NA>,<NA>,103,<NA>,112
2513,E. Norby,1335271936,ECU,1017,L,119,111,<NA>,95,131,100,<NA>,112
190,S. Jenkins,1058441697,ECU,505,R,105,113,<NA>,110,111,<NA>,<NA>,111
1255,R. Towers,1196825344,ECU,502,L,113,<NA>,105,<NA>,<NA>,109,<NA>,111
3767,T. Paxton,1684934913,ECU,31,R,110,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,110
1519,G. Marley,1225307968,ECU,454,R,109,93,<NA>,<NA>,<NA>,<NA>,<NA>,108
808,E. Rose,1147995857,ECU,468,R,107,87,<NA>,<NA>,112,<NA>,<NA>,106
2279,A. Bouche,1310415616,ECU,140,R,102,111,109,<NA>,<NA>,<NA>,<NA>,106
